In [1]:
# Cluster-compatible setup
import os, torch

BASE_PATH = os.getcwd()
DATA_PATH = os.path.join(BASE_PATH, 'data')
OUTPUT_PATH = os.path.join(BASE_PATH, 'outputs')
os.makedirs(OUTPUT_PATH, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Using device: cuda
GPU: NVIDIA RTX A5000


# AIC Cable Insertion — ACT Policy Training

Trains an **ACT (Action Chunking Transformer)** policy on your HuggingFace dataset using a Colab GPU.  
Checkpoints are saved to **Google Drive** after every `SAVE_FREQ` steps so you can resume any time.

---

## Before you start

1. **Runtime → Change runtime type → T4 GPU** (free tier) or A100 (Colab Pro)
2. Your dataset must be uploaded to HuggingFace Hub  
   (`pixi run python my_policy_node/scripts/push_dataset_to_hub.py ...`)
3. You need a HuggingFace account (free) — login prompt appears in Step 5

---

## Quick-start

| Step | What it does |
|------|--------------|
| 1 | Check GPU |
| 2 | Mount Google Drive (for persistent checkpoints) |
| 3 | Install LeRobot |
| 4 | **Fill in your config** (HF username, dataset, etc.) |
| 5 | HuggingFace login |
| 6 | Preview dataset features |
| 7 | **Run training** |
| 8 | Resume after disconnection |
| 9 | Inspect checkpoints |

---
## Step 1 — Check GPU

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Go to  Runtime → Change runtime type → Hardware accelerator → T4 GPU"
    )

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {name}")
print(f"VRAM : {vram:.1f} GB")
print(f"CUDA : {torch.version.cuda}")

GPU  : NVIDIA RTX A5000
VRAM : 25.3 GB
CUDA : 12.8


---
## Step 2 — Mount Google Drive

Checkpoints are written to `My Drive/aic_training/<dataset>_act/`.  
They persist across Colab sessions so you can always resume.

---
## Step 3 — Install LeRobot

Pins to **v0.5.1** to match the version used for data collection.  
This cell takes ~2 minutes on first run.

In [ ]:
# The cluster has conda numpy 2.x, but conda-installed pandas was compiled
# against numpy 1.x (ABI mismatch → ImportError on pandas._libs).
# Fix: reinstall pandas via pip to get a wheel built for numpy 2.x.
# --no-deps avoids reinstalling numpy itself (conda owns it).
!pip install -q "pandas>=2.2.0" --force-reinstall --no-deps
!pip install -q "lerobot==0.4.4" wandb

import lerobot
import numpy as np
import pandas as pd
print(f"LeRobot {lerobot.__version__}")
print(f"NumPy   {np.__version__}")
print(f"Pandas  {pd.__version__}")

---
## Step 4 — Configuration

**Edit the values in this cell before running anything else.**

In [ ]:
# ============================================================
#  YOUR SETTINGS — edit these
# ============================================================

HF_USERNAME   = "Nikhil-Ravi"            # @param {type:"string"}
DATASET_NAME  = "sfp_insertion_demos"    # @param {type:"string"}

# Training duration
STEPS         = 100_000   # @param {type:"integer"}   total gradient steps
BATCH_SIZE    = 8         # @param {type:"integer"}   reduce to 4 if OOM

# ACT action chunk — how many future actions to predict at once.
CHUNK_SIZE    = 50        # @param {type:"integer"}

# Checkpointing — save every SAVE_FREQ steps
SAVE_FREQ     = 2_000     # @param {type:"integer"}

# Weights & Biases (optional — leave empty to skip)
WANDB_PROJECT = "Insertion_demos"        # @param {type:"string"}

# ============================================================
#  Derived — do not edit
# ============================================================

DATASET_REPO_ID = f"{HF_USERNAME}/{DATASET_NAME}"
CHECKPOINT_DIR  = os.path.join(DATA_PATH, f"aic_training/{DATASET_NAME}_act")
USE_WANDB       = bool(WANDB_PROJECT)

print(f"Dataset      : {DATASET_REPO_ID}")
print(f"Steps        : {STEPS:,}   Batch: {BATCH_SIZE}   Chunk: {CHUNK_SIZE}")
print(f"Save every   : {SAVE_FREQ:,} steps")
print(f"Checkpoint   : {CHECKPOINT_DIR}")
print(f"W&B          : {'enabled — project=' + WANDB_PROJECT if USE_WANDB else 'disabled'}")

---
## Step 5 — HuggingFace Login

Needed to download your (private) dataset.  
Get a token at https://huggingface.co/settings/tokens (read access is enough).

In [5]:
from huggingface_hub import notebook_login
notebook_login()

---
## Step 6 — Preview Dataset

Shows the features present in the dataset and builds the policy input/output config.

In [6]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.configs.types import FeatureType, PolicyFeature

print(f"Loading dataset metadata from {DATASET_REPO_ID} ...")
ds = LeRobotDataset(DATASET_REPO_ID)

print(f"\nEpisodes : {ds.num_episodes}")
print(f"Frames   : {len(ds):,}")
print(f"FPS      : {ds.fps}")
print(f"\nFeatures:")
for k, v in sorted(ds.features.items()):
    print(f"  {k:<50s}  shape={str(v['shape']):<15s}  dtype={v['dtype']}")

# ---------- Build PolicyFeature dicts from dataset metadata ----------
input_features  = {}
output_features = {}

for key, feat in ds.features.items():
    shape = tuple(feat["shape"])
    if key.startswith("observation.images"):
        input_features[key] = PolicyFeature(type=FeatureType.VISUAL, shape=shape)
    elif key.startswith("observation."):
        input_features[key] = PolicyFeature(type=FeatureType.STATE, shape=shape)
    elif key == "action":
        output_features[key] = PolicyFeature(type=FeatureType.ACTION, shape=shape)

print(f"\nPolicy inputs  ({len(input_features)} features):")
for k, v in input_features.items():
    print(f"  {v.type.value:<8s}  {k}  {v.shape}")
print(f"\nPolicy outputs ({len(output_features)} features):")
for k, v in output_features.items():
    print(f"  {v.type.value:<8s}  {k}  {v.shape}")

Loading dataset metadata from Nikhil-Ravi/sfp_insertion_demos ...


Fetching 39 files: 100%|██████████| 39/39 [00:08<00:00,  4.79it/s]



Episodes : 50
Frames   : 9,353
FPS      : 10

Features:
  action                                              shape=(7,)             dtype=float32
  episode_index                                       shape=(1,)             dtype=int64
  frame_index                                         shape=(1,)             dtype=int64
  index                                               shape=(1,)             dtype=int64
  observation.images.center_camera                    shape=(3, 256, 288)    dtype=image
  observation.images.left_camera                      shape=(3, 256, 288)    dtype=image
  observation.images.right_camera                     shape=(3, 256, 288)    dtype=image
  observation.state                                   shape=(26,)            dtype=float32
  observation.wrist_force                             shape=(3,)             dtype=float32
  task_index                                          shape=(1,)             dtype=int64
  timestamp                                    

## Check for checkpoint

In [ ]:
import os
import re
from pathlib import Path

def find_latest_checkpoint(checkpoint_dir):
    ckpt_root = os.path.join(checkpoint_dir, "checkpoints")
    if not os.path.exists(ckpt_root):
        return None, None
    steps = [int(n) for n in os.listdir(ckpt_root) if re.match(r"^\d+$", n)]
    if not steps:
        return None, None
    latest_step = max(steps)
    latest_path = os.path.join(ckpt_root, f"{latest_step:06d}")
    return latest_step, latest_path

latest_step, latest_path = find_latest_checkpoint(CHECKPOINT_DIR)

if latest_step is not None:
    print(f"Resume available: YES")
    print(f"Resuming from step: {latest_step:,}")
    print(f"Checkpoint path  : {latest_path}")
    RESUME = True
else:
    print("Resume available: NO (training will start from scratch)")
    RESUME = False

---
## Step 7 — Train

- If `CHECKPOINT_DIR` already contains a checkpoint from a previous run, training **resumes automatically**.
- Checkpoints are saved to Drive every `SAVE_FREQ` steps.
- Expected time on a T4 GPU: ~2–3 hours for 100 k steps.

> **Tip:** If the session disconnects, just re-run Steps 1–6 and then this cell — it will pick up where it left off.

In [ ]:
from pathlib import Path
from datetime import datetime

# ── Patch: inject missing 'names' for image features ─────────────────────
import lerobot.policies.factory as _pf

try:
    import lerobot.datasets.utils as _du
    _patch_target = _du
except ImportError:
    try:
        import lerobot.datasets.feature_utils as _du
        _patch_target = _du
    except ImportError:
        _patch_target = None

if _patch_target is not None and hasattr(_patch_target, "dataset_to_policy_features"):
    _current = _patch_target.dataset_to_policy_features
    if getattr(_current, "__name__", "") != "_d2pf_no_names":
        # First time — wrap the real original
        def _d2pf_no_names(features, _fn=_current):
            fixed = {}
            for k, v in features.items():
                v = dict(v)
                if len(v.get("shape", [])) == 3 and "names" not in v:
                    v["names"] = ["channel", "height", "width"]
                fixed[k] = v
            return _fn(fixed)
        _patch_target.dataset_to_policy_features = _d2pf_no_names
        _pf.dataset_to_policy_features = _d2pf_no_names
        print(f"Patched dataset_to_policy_features in {_patch_target.__name__}")
    else:
        print("dataset_to_policy_features already patched — skipping")

# ── Fresh run in a timestamped directory ──────────────────────────────────
from lerobot.configs.default import DatasetConfig
from lerobot.configs.train import TrainPipelineConfig
from lerobot.configs.types import NormalizationMode
from lerobot.policies.act.configuration_act import ACTConfig

try:
    from lerobot.scripts.lerobot_train import train
except ImportError:
    from lerobot.scripts.train import train

RUN_TAG   = datetime.now().strftime("%Y%m%d_%H%M%S")
ckpt_root = Path(DATA_PATH) / "aic_training" / f"{DATASET_NAME}_act_{RUN_TAG}"
print(f"Output dir : {ckpt_root}")

policy_cfg = ACTConfig(
    input_features  = input_features,
    output_features = output_features,
    normalization_mapping = {
        "VISUAL" : NormalizationMode.MEAN_STD,
        "STATE"  : NormalizationMode.MEAN_STD,
        "ACTION" : NormalizationMode.MEAN_STD,
    },
    chunk_size      = CHUNK_SIZE,
    n_action_steps  = CHUNK_SIZE,
    n_obs_steps     = 1,
    dim_model       = 256,
    n_heads         = 8,
    dim_feedforward = 3200,
    n_encoder_layers= 4,
    n_decoder_layers= 1,
    use_vae         = True,
    latent_dim      = 32,
    kl_weight       = 10.0,
)

train_cfg = TrainPipelineConfig(
    dataset        = DatasetConfig(repo_id=DATASET_REPO_ID, episodes=None),
    policy         = policy_cfg,
    output_dir     = ckpt_root,
    resume         = False,
    steps          = STEPS,
    batch_size     = BATCH_SIZE,
    num_workers    = 4,
    eval_freq      = -1,
    log_freq       = 200,
    save_checkpoint= True,
    save_freq      = SAVE_FREQ,
    seed           = 42,
)

try:
    train_cfg.policy.push_to_hub = False
except AttributeError:
    pass

if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT, name=f"{DATASET_NAME}_act_{RUN_TAG}", resume="never")
    try:
        train_cfg.wandb.enable  = True
        train_cfg.wandb.project = WANDB_PROJECT
    except AttributeError:
        pass

print(f"Starting training ({STEPS:,} steps, saving every {SAVE_FREQ:,})...")
train(train_cfg)

---
## Step 8 — Resume after disconnection

If your Colab session disconnected or you closed the browser:

1. Open this notebook again
2. Run **Steps 1 → 6** in order (GPU check, Drive mount, install, config, login, dataset preview)
3. Run the cell below **instead of** Step 7

It loads the latest checkpoint from Drive and continues from where it left off.

In [ ]:
# ── Resume cell ───────────────────────────────────────────────────────────
# lerobot 0.5.x resume bugs we work around:
#   1. validate() sets policy.pretrained_path = output_dir (run root).
#   2. lerobot_train.py imports make_policy / load_training_state locally at
#      load time — patching the source modules is a no-op; must patch the
#      names in lerobot_train's own namespace.
#   3. TrainPipelineConfig.optimizer is None after JSON round-trip.
#   4. cfg.checkpoint_path points to the wrong directory.

import sys
import json
import dataclasses
from pathlib import Path

# ── Names patch ──────────────────────────────────────────────────────────
import lerobot.policies.factory as _pf

try:
    import lerobot.datasets.utils as _du
    _patch_target = _du
except ImportError:
    try:
        import lerobot.datasets.feature_utils as _du
        _patch_target = _du
    except ImportError:
        _patch_target = None

if _patch_target is not None and hasattr(_patch_target, "dataset_to_policy_features"):
    _current = _patch_target.dataset_to_policy_features
    if getattr(_current, "__name__", "") != "_d2pf_no_names":
        def _d2pf_no_names(features, _fn=_current):
            fixed = {}
            for k, v in features.items():
                v = dict(v)
                if len(v.get("shape", [])) == 3 and "names" not in v:
                    v["names"] = ["channel", "height", "width"]
                fixed[k] = v
            return _fn(fixed)
        _patch_target.dataset_to_policy_features = _d2pf_no_names
        _pf.dataset_to_policy_features = _d2pf_no_names
        print("Patched dataset_to_policy_features")

# ── Find the most recent run and its latest checkpoint ───────────────────
training_base = Path(DATA_PATH) / "aic_training"
run_dirs = sorted(
    [d for d in training_base.glob(f"{DATASET_NAME}_act*") if d.is_dir()],
    key=lambda d: d.stat().st_mtime,
)
if not run_dirs:
    raise RuntimeError(f"No run directories found under {training_base}")

ckpt_root       = run_dirs[-1]
checkpoint_dirs = sorted(ckpt_root.glob("checkpoints/*/"))
if not checkpoint_dirs:
    raise RuntimeError(f"No checkpoints found under {ckpt_root / 'checkpoints'}")

latest_ckpt_dir   = checkpoint_dirs[-1]           # e.g. .../checkpoints/050000/
latest_pretrained = latest_ckpt_dir / "pretrained_model"

print(f"Run              : {ckpt_root.name}")
print(f"Checkpoints      : {len(checkpoint_dirs)}  (latest step: {latest_ckpt_dir.name})")
print(f"Pretrained model : {latest_pretrained}")
print(f"Training state   : {latest_ckpt_dir / 'training_state'}")

if not (latest_pretrained / "model.safetensors").is_file():
    raise RuntimeError(f"model.safetensors not found at {latest_pretrained}")

# ── Helper: redirect run-root pretrained_path to checkpoint subdir ───────
def _fix_pretrained_path(p_str):
    p = Path(str(p_str))
    if p.is_dir() and not (p / "model.safetensors").is_file():
        candidates = sorted(p.glob("checkpoints/*/pretrained_model/model.safetensors"))
        if candidates:
            correct = str(candidates[-1].parent)
            print(f"[ckpt-fix] pretrained_path -> .../{candidates[-1].parent.parent.name}/pretrained_model/")
            return correct
    return p_str

# ── Patch 1: TrainPipelineConfig.validate ────────────────────────────────
from lerobot.configs.train import TrainPipelineConfig as _TPC

if getattr(_TPC.validate, "__name__", "") != "_patched_validate":
    _orig_validate = _TPC.validate

    def _patched_validate(self, _fn=_orig_validate):
        _fn(self)
        if not getattr(self, "resume", False):
            return
        # Fix policy.pretrained_path
        if hasattr(self, "policy"):
            pp = getattr(self.policy, "pretrained_path", None)
            if pp:
                fixed = _fix_pretrained_path(pp)
                if fixed != pp:
                    try:
                        self.policy.pretrained_path = fixed
                    except Exception:
                        try:
                            object.__setattr__(self.policy, "pretrained_path", fixed)
                        except Exception as e:
                            print(f"[ckpt-fix] Warning pretrained_path: {e}")
        # Best-effort fix of checkpoint_path (may fail if dataclass is frozen)
        cp_before = getattr(self, "checkpoint_path", None)
        try:
            self.checkpoint_path = latest_ckpt_dir
        except Exception:
            try:
                object.__setattr__(self, "checkpoint_path", latest_ckpt_dir)
            except Exception:
                pass
        cp_after = getattr(self, "checkpoint_path", None)
        if cp_after != latest_ckpt_dir:
            print(f"[ckpt-fix] WARNING: could not set checkpoint_path on config "
                  f"(before={cp_before}, after={cp_after}) — Patch 3 will fix it")
        else:
            print(f"[ckpt-fix] checkpoint_path -> .../checkpoints/{latest_ckpt_dir.name}/")

    _patched_validate.__name__ = "_patched_validate"
    _TPC.validate = _patched_validate
    print("Patched TrainPipelineConfig.validate")

# ── Patch 2: lerobot_train.make_policy ───────────────────────────────────
try:
    import lerobot.scripts.lerobot_train as _llt

    if getattr(getattr(_llt, "make_policy", None), "__name__", "") != "_make_policy_ckpt_fix":
        _orig_llt_mp = _llt.make_policy

        def _make_policy_ckpt_fix(cfg, ds_meta=None, env_cfg=None, rename_map=None, **extra):
            pp = getattr(cfg, "pretrained_path", None)
            if pp:
                fixed = _fix_pretrained_path(pp)
                if fixed != pp:
                    try:
                        cfg.pretrained_path = fixed
                    except Exception:
                        try:
                            object.__setattr__(cfg, "pretrained_path", fixed)
                        except Exception:
                            pass
            kw = {}
            if ds_meta    is not None: kw["ds_meta"]    = ds_meta
            if env_cfg    is not None: kw["env_cfg"]    = env_cfg
            if rename_map is not None: kw["rename_map"] = rename_map
            kw.update(extra)
            return _orig_llt_mp(cfg, **kw)

        _make_policy_ckpt_fix.__name__ = "_make_policy_ckpt_fix"
        _llt.make_policy = _make_policy_ckpt_fix
        print("Patched lerobot_train.make_policy")

    # ── Patch 3: lerobot_train.load_training_state ───────────────────────
    # cfg.checkpoint_path may not be patchable on the config object itself.
    # Intercept the call and redirect the path directly if training_state is absent.
    if getattr(getattr(_llt, "load_training_state", None), "__name__", "") != "_load_ts_fix":
        _orig_lts = _llt.load_training_state

        def _load_ts_fix(checkpoint_dir, optimizer, scheduler=None):
            p = Path(str(checkpoint_dir))
            if not (p / "training_state").is_dir():
                print(f"[ckpt-fix] load_training_state: no training_state in {p.name}/,"
                      f" redirecting to checkpoints/{latest_ckpt_dir.name}/")
                p = latest_ckpt_dir
            return _orig_lts(p, optimizer, scheduler)

        _load_ts_fix.__name__ = "_load_ts_fix"
        _llt.load_training_state = _load_ts_fix
        print("Patched lerobot_train.load_training_state")

except ImportError:
    pass

# ── Build config ──────────────────────────────────────────────────────────
from lerobot.configs.default import DatasetConfig
from lerobot.configs.train import TrainPipelineConfig
from lerobot.configs.types import NormalizationMode
from lerobot.policies.act.configuration_act import ACTConfig

try:
    from lerobot.scripts.lerobot_train import train
except ImportError:
    from lerobot.scripts.train import train

policy_cfg = ACTConfig(
    input_features   = input_features,
    output_features  = output_features,
    normalization_mapping = {
        "VISUAL" : NormalizationMode.MEAN_STD,
        "STATE"  : NormalizationMode.MEAN_STD,
        "ACTION" : NormalizationMode.MEAN_STD,
    },
    chunk_size       = CHUNK_SIZE,
    n_action_steps   = CHUNK_SIZE,
    n_obs_steps      = 1,
    dim_model        = 256,
    n_heads          = 8,
    dim_feedforward  = 3200,
    n_encoder_layers = 4,
    n_decoder_layers = 1,
    use_vae          = True,
    latent_dim       = 32,
    kl_weight        = 10.0,
)

train_cfg = TrainPipelineConfig(
    dataset        = DatasetConfig(repo_id=DATASET_REPO_ID),
    policy         = policy_cfg,
    output_dir     = ckpt_root,
    resume         = True,
    steps          = STEPS,
    batch_size     = BATCH_SIZE,
    num_workers    = 4,
    eval_freq      = -1,
    log_freq       = 200,
    save_checkpoint= True,
    save_freq      = SAVE_FREQ,
    seed           = 42,
)

try:
    train_cfg.policy.push_to_hub = False
except AttributeError:
    pass

try:
    train_cfg.checkpoint_path = latest_ckpt_dir
    print(f"Set checkpoint_path: checkpoints/{latest_ckpt_dir.name}/")
except (AttributeError, TypeError):
    pass

# ── Add optimizer config (required in lerobot 0.5.x) ─────────────────────
if getattr(train_cfg, "optimizer", None) is None:
    _opt_set = False
    for _mod_name, _cls_name in [
        ("lerobot.optim.optimizers", "AdamWConfig"),
        ("lerobot.configs.default",  "AdamWConfig"),
    ]:
        try:
            import importlib
            _m = importlib.import_module(_mod_name)
            _AdamWCls = getattr(_m, _cls_name)
            try:
                train_cfg.optimizer = _AdamWCls(
                    lr=1e-5, betas=(0.95, 0.999), eps=1e-8,
                    weight_decay=1e-4, grad_clip_norm=10.0,
                )
            except TypeError:
                train_cfg.optimizer = _AdamWCls(
                    lr=1e-5, betas=(0.95, 0.999), eps=1e-8, weight_decay=1e-4,
                )
            print(f"Set optimizer: AdamW(lr=1e-5)  [{_mod_name}]")
            _opt_set = True
            break
        except (ImportError, AttributeError, TypeError):
            pass
    if not _opt_set:
        print("Warning: could not locate AdamWConfig — training will likely fail")
else:
    print(f"Optimizer already set: {type(train_cfg.optimizer).__name__}")

# ── Write / refresh train_config.json ────────────────────────────────────
train_config_json = ckpt_root / "train_config.json"
try:
    import draccus
    with open(train_config_json, "w") as f:
        draccus.dump(train_cfg, f)
    print(f"Wrote train_config.json (draccus)")
except Exception as e:
    with open(train_config_json, "w") as f:
        json.dump(dataclasses.asdict(train_cfg), f, indent=2, default=str)
    print(f"Wrote train_config.json (json fallback; draccus: {e})")

sys.argv = [sys.argv[0], f"--config_path={train_config_json}"]

# ── Resume ────────────────────────────────────────────────────────────────
if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT, name=ckpt_root.name, resume="allow")
    try:
        train_cfg.wandb.enable  = True
        train_cfg.wandb.project = WANDB_PROJECT
    except AttributeError:
        pass

print(f"\nResuming from step {latest_ckpt_dir.name} ({STEPS:,} steps total, saving every {SAVE_FREQ:,})...")
train(train_cfg)

---
## Step 9 — Download trained policy

Zips the latest checkpoint's `pretrained_model/` folder and gives you a download link.

In [ ]:
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

# ── Find latest pretrained_model ─────────────────────────────────────────
training_base = Path(DATA_PATH) / "aic_training"
run_dirs = sorted(
    [d for d in training_base.glob(f"{DATASET_NAME}_act*") if d.is_dir()],
    key=lambda d: d.stat().st_mtime,
)
if not run_dirs:
    raise RuntimeError(f"No run directories found under {training_base}")

ckpt_root       = run_dirs[-1]
checkpoint_dirs = sorted(ckpt_root.glob("checkpoints/*/"))
if not checkpoint_dirs:
    raise RuntimeError(f"No checkpoints found under {ckpt_root / 'checkpoints'}")

latest_ckpt_dir   = checkpoint_dirs[-1]
pretrained_dir    = latest_ckpt_dir / "pretrained_model"

if not pretrained_dir.is_dir():
    raise RuntimeError(f"pretrained_model not found at {pretrained_dir}")

# ── Zip it up ────────────────────────────────────────────────────────────
zip_name = f"{ckpt_root.name}_step{latest_ckpt_dir.name}.zip"
zip_path = Path(DATA_PATH) / zip_name

print(f"Zipping {pretrained_dir} ...")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(pretrained_dir.rglob("*")):
        if f.is_file():
            arcname = f.relative_to(pretrained_dir.parent)
            zf.write(f, arcname)
            print(f"  + {arcname}  ({f.stat().st_size / 1e6:.1f} MB)")

size_mb = zip_path.stat().st_size / 1e6
print(f"\nCreated: {zip_path.name}  ({size_mb:.1f} MB)")
print("Click the link below to download:")
display(FileLink(str(zip_path.relative_to(Path.cwd()))))

---
## Step 9 — Inspect checkpoints

Lists all saved checkpoints and their sizes.

In [ ]:
from pathlib import Path

ckpt_root = Path(CHECKPOINT_DIR) / "checkpoints"

if not ckpt_root.exists():
    print("No checkpoints yet — run Step 7 first.")
else:
    ckpts = sorted(ckpt_root.iterdir())
    print(f"{'Checkpoint':<30} {'Size':>10}")
    print("-" * 42)
    total = 0
    for c in ckpts:
        size = sum(f.stat().st_size for f in c.rglob("*") if f.is_file())
        total += size
        print(f"{c.name:<30} {size/1e6:>9.1f} MB")
    print("-" * 42)
    print(f"{'Total:':<30} {total/1e6:>9.1f} MB  ({len(ckpts)} checkpoints)")
    print(f"\nDrive path: {ckpt_root}")

---
## Troubleshooting

**Out of memory (CUDA OOM)**  
Reduce `BATCH_SIZE` to 4 in Step 4, then re-run Steps 6–7.

**`eval_freq` error / environment not found**  
Make sure `eval_freq = -1` in the training config. Evaluation requires a simulation environment which is not available in Colab.

**`KeyError` on a feature name**  
Step 6 must complete before Step 7 — it builds `input_features` and `output_features` from your dataset. If the kernel restarted, re-run Steps 1–6.

**Training loss not decreasing after 20k steps**  
- Try lowering `CHUNK_SIZE` to 25 (shorter prediction horizon)
- Increase `BATCH_SIZE` to 16 if VRAM allows
- Collect more episodes (aim for 100+)

**W&B not logging**  
Run `!wandb login` in a new cell and paste your API key from https://wandb.ai/settings